# 04: Sensitivity analysis and correlation

**What:** Stress-test our own SQL results, then check one relationship (volume vs failure rate).
**Why:** Every result rests on choices we made (rounded rates, tier cut-offs, medians, which banks are separate). If a finding changes when we change a choice, it is not safe for the memo or resume.

**Inputs:** `upi_remitter_mapped.csv` (1,198 rows), `bank_mapping.csv` (117 rows)
**Output:** a verdict table, `sensitivity_summary.csv`

| Check | What we change | Question |
|---|---|---|
| S1 | Move every published rate by its rounding error (±0.005 pp) | Does the year-on-year improvement survive? |
| S2 | Merge the 3 "possible same bank" pairs | Do the watchlist and concentration change? |
| S3 | Include or exclude the 6 unranked entries | Does the trend change? |
| S4 | Use a gentler target in the what-if | How much of the "62% avoidable" survives? |
| S5 | Require a minimum rate before a bank can be "Episodic" | Is that label an artifact of tiny rates? |
| 8 | Spearman correlation, month-on-month changes | Do volume jumps go with failure-rate jumps? |

**Rules:** changes are in percentage points (pp). Findings say "associated with", never "caused by".

## Step 0: Load and validate
**What:** Read the two CSVs and run checks that stop the notebook if the data is not what we expect.

**Why:** Fail loudly now rather than produce wrong numbers later.

In [3]:
import pandas as pd, numpy as np
from scipy.stats import spearmanr

try:                                    # Colab: upload the two CSVs when asked
    from google.colab import files
    files.upload()                      # choose upi_remitter_mapped.csv and bank_mapping.csv
    PATH = ''
except ImportError:                     # local run (VS Code): files sit in the clean folder
    PATH = '../clean/'

df = pd.read_csv(PATH + 'upi_remitter_mapped.csv', parse_dates=['month'])
mp = pd.read_csv(PATH + 'bank_mapping.csv')

df['est_failed'] = df['volume_mn'] * df['td_pct'] / 100     # estimated failed payments, millions

assert len(df) == 1198 and df['month'].nunique() == 24
assert df[['volume_mn', 'bd_pct', 'td_pct']].notna().all().all()
assert not df.duplicated(['month', 'bank_name']).any()
print('Loaded:', df.shape, '| months:', df['month'].nunique(), '| banks:', df['bank_name'].nunique())

Saving upi_remitter_mapped.csv to upi_remitter_mapped (1).csv
Loaded: (1198, 8) | months: 24 | banks: 61


In [5]:
df

,month,bank_name,volume_mn,approved_pct,bd_pct,td_pct,is_ranked,est_failed
0,2024-08-01,AU Small Finance Bank,69.400000,91.26,8.72,0.02,1,0.013880
1,2024-08-01,Airtel Payments Bank Limited,437.930000,90.01,9.84,0.15,1,0.656895
2,2024-08-01,Andhra Pradesh Grameena Vikas Bank,28.610000,86.25,12.84,0.91,1,0.260351
3,2024-08-01,Andhra Pragathi Grameena Bank,17.780000,85.45,12.94,1.61,1,0.286258
4,2024-08-01,Axis Bank Ltd.,730.470000,95.47,4.53,0.00,1,0.000000
...,...,...,...,...,...,...,...,...
1193,2026-07-01,Tri O Tech Solutions Private Limited,202.881109,100.00,0.00,0.00,0,0.000000
1194,2026-07-01,UCO Bank,359.225624,87.06,12.86,0.09,1,0.323303
1195,2026-07-01,Ujjivan Small Finance Bank Limited,68.946963,85.65,14.01,0.33,1,0.227525
1196,2026-07-01,Union Bank of India,1544.225883,89.60,10.39,0.01,1,0.154423


## Step 1: Reconcile with the SQL result (Q1)
**What:** Recompute the monthly weighted TD rate in pandas and compare with the MySQL output.

**Why:** If Python and SQL disagree, one of them is wrong, and we must know before testing anything.

Weighted TD rate = total estimated failed / total volume (big banks count more).

In [6]:
monthly = df.groupby('month').agg(volume=('volume_mn', 'sum'), failed=('est_failed', 'sum'))
monthly['td_rate'] = monthly['failed'] / monthly['volume'] * 100

sql_values = {'2024-08-01': 0.354, '2024-10-01': 0.671, '2026-02-01': 0.202, '2026-07-01': 0.399}
for m, v in sql_values.items():
    assert round(monthly.loc[m, 'td_rate'], 3) == v, f'Mismatch with SQL at {m}'

def period_rate(d):
    return d['est_failed'].sum() / d['volume_mn'].sum() * 100

months = sorted(df['month'].unique())
p1, p2 = months[:12], months[12:]               # first 12 and last 12 months
y1 = period_rate(df[df['month'].isin(p1)])
y2 = period_rate(df[df['month'].isin(p2)])
print(f'Matches SQL on 4 test months.  Year 1: {y1:.3f}%   Year 2: {y2:.3f}%')

Matches SQL on 4 test months.  Year 1: 0.432%   Year 2: 0.324%


## Step 2: Rebuild the SQL logic as functions

**What:** Recreate our eligible banks, size tiers, peer medians, profile types, the what-if, and the priority matrix as reusable functions.

**Why:** The sensitivity checks re-run the same logic on changed inputs. Writing it once avoids copy-paste errors. We then prove the functions equal the SQL by matching your verified numbers.


- `build_bank_month`: eligible banks (ranked, 12+ months), tier by average volume (Large 500+, Mid 100 to 500, Small below 100), and the tier median TD rate per month
- `profile`: persistence, volatility (CV), and type: Chronic, Mixed, Episodic, Healthy
- `prize`: failures avoidable if bank-months above `k` x their tier median came down to it
- `matrix`: impact vs severity quadrants

In [7]:
def build_bank_month(d, min_months=12):
    r = d[d['is_ranked'] == 1]
    info = r.groupby('bank_name').agg(months=('month', 'nunique'),
                                      avg_volume_mn=('volume_mn', 'mean')).reset_index()
    info = info[info['months'] >= min_months].copy()
    info['tier'] = np.select([info['avg_volume_mn'] >= 500, info['avg_volume_mn'] >= 100],
                             ['Large', 'Mid'], default='Small')
    bm = r.merge(info[['bank_name', 'tier']], on='bank_name')
    med = bm.groupby(['month', 'tier'])['td_pct'].median().rename('tier_median_td').reset_index()
    return bm.merge(med, on=['month', 'tier'])

def profile(bm, floor=0.0):
    g = bm.assign(above=bm['td_pct'] > bm['tier_median_td']).groupby(['bank_name', 'tier'])
    p = g.agg(months=('month', 'count'), failed=('est_failed', 'sum'), vol=('volume_mn', 'sum'),
              persistence=('above', 'mean'), mean_td=('td_pct', 'mean'),
              sd_td=('td_pct', lambda s: s.std(ddof=0))).reset_index()
    p['persistence'] = p['persistence'] * 100
    p['cv'] = p['sd_td'] / p['mean_td']
    p['weighted_td'] = p['failed'] / p['vol'] * 100
    p['type'] = np.select([p['persistence'] >= 75,
                           p['persistence'] > 50,
                           (p['cv'] >= 1.15) & (p['mean_td'] >= floor)],
                          ['Chronic', 'Mixed', 'Episodic'], default='Healthy')
    return p

def prize(bm, k=1.0):
    excess = (bm['td_pct'] - k * bm['tier_median_td']).clip(lower=0) * bm['volume_mn'] / 100
    return excess.sum(), excess.sum() / bm['est_failed'].sum() * 100

def matrix(bm):
    b = bm.groupby(['bank_name', 'tier']).agg(failed=('est_failed', 'sum'),
                                              vol=('volume_mn', 'sum')).reset_index()
    t = b.groupby('tier').apply(lambda x: x['failed'].sum() / x['vol'].sum(),
                                include_groups=False).rename('tier_rate')
    b = b.merge(t, on='tier')
    b['impact_pct'] = b['failed'] / b['failed'].sum() * 100
    b['severity_x'] = (b['failed'] / b['vol']) / b['tier_rate']
    hi_i = b['impact_pct'] >= 100 / len(b)           # at least an "equal share"
    hi_s = b['severity_x'] >= 1.25                   # at least 1.25 x tier rate
    b['quadrant'] = np.select([hi_i & hi_s, hi_i, hi_s],
                              ['1 Fix first', '2 Big but in line', '3 Small but weak'], default='4 Healthy')
    return b

# ---- prove the functions equal the verified SQL results ----
bm = build_bank_month(df)
tiers = bm.groupby('tier')['bank_name'].nunique().to_dict()
assert tiers == {'Large': 13, 'Mid': 11, 'Small': 25} and len(bm) == 1135

base_profile = profile(bm)
counts = base_profile['type'].value_counts().to_dict()
assert counts == {'Healthy': 19, 'Chronic': 13, 'Episodic': 9, 'Mixed': 8}

avoidable, pct = prize(bm)
assert round(avoidable, 1) == 1118.3 and round(pct, 1) == 62.1

base_matrix = matrix(bm)
watch = sorted(base_matrix.loc[base_matrix['quadrant'] == '1 Fix first', 'bank_name'])
print('Tiers:', tiers, '| rows:', len(bm))
print('Profile types:', counts)
print(f'Prize: {avoidable:.1f} Mn = {pct:.1f}% of failures')
print('Fix-first banks:', watch)

Tiers: {'Large': 13, 'Mid': 11, 'Small': 25} | rows: 1135
Profile types: {'Healthy': 19, 'Chronic': 13, 'Episodic': 9, 'Mixed': 8}
Prize: 1118.3 Mn = 62.1% of failures
Fix-first banks: ['Bank of India', 'Baroda U.P. Bank', 'Central Bank of India', 'India Post Payments Bank Limited', 'Indian Overseas Bank', 'State Bank of India']


## S1: Rounding bounds

**What:** NPCI publishes rates rounded to 2 decimals, so each true rate can be up to 0.005 pp higher or lower. We shift every rate down and up by 0.005 and see how far our results can move.

**Why:** A small rate like 0.03% has a large relative rounding error. We must know whether our headline results depend on it.

**Two tests:**
1. Year 1 vs year 2 weighted rate: do the ranges still stay apart?
2. Fix-first banks: even in the worst case (bank rate at its lowest, tier rate at its highest), are they still 1.25x or more above their tier?

In [8]:
def bounds(d, delta=0.005):
    v = d['volume_mn'].sum()
    lo = (d['volume_mn'] * (d['td_pct'] - delta).clip(lower=0) / 100).sum() / v * 100
    hi = (d['volume_mn'] * (d['td_pct'] + delta) / 100).sum() / v * 100
    return lo, hi

a_lo, a_hi = bounds(df[df['month'].isin(p1)])
b_lo, b_hi = bounds(df[df['month'].isin(p2)])
print(f'Year 1 range: {a_lo:.3f} to {a_hi:.3f}   Year 2 range: {b_lo:.3f} to {b_hi:.3f}')
print('Improvement survives rounding:', a_lo > b_hi)

rows = []
for name in watch:
    one = bm[bm['bank_name'] == name]
    peers = bm[bm['tier'] == one['tier'].iloc[0]]
    lo, _ = bounds(one)
    _, thi = bounds(peers)
    rows.append((name, one['tier'].iloc[0], round(lo, 3), round(thi, 3), round(lo / thi, 2)))
s1 = pd.DataFrame(rows, columns=['bank', 'tier', 'bank_rate_low', 'tier_rate_high', 'worst_case_severity_x'])
s1

Year 1 range: 0.427 to 0.437   Year 2 range: 0.319 to 0.329
Improvement survives rounding: True


,bank,tier,bank_rate_low,tier_rate_high,worst_case_severity_x
0,Bank of India,Large,0.742,0.320,2.32
1,Baroda U.P. Bank,Small,4.129,1.153,3.58
2,Central Bank of India,Mid,0.850,0.370,2.30
3,India Post Payments Bank Limited,Large,0.723,0.320,2.26
4,Indian Overseas Bank,Mid,0.577,0.370,1.56
5,State Bank of India,Large,0.480,0.320,1.50


## S2: Merge the 3 "possible same bank" pairs

**What:** `bank_mapping.csv` marks 3 short-lived names that may be older names of ranked banks (Andhra Pradesh Grameena, Rajasthan Gramin, Uttar Pradesh Gramin). We merge each into its partner and re-run.

**Why:** We kept them separate because we could not prove they were the same bank. If merging them changes our conclusions, that choice matters.

**How:** rename, then combine any overlapping month with a volume-weighted average (a check confirms total volume is unchanged).

**Compare:** Top 10 share of failures, the Fix-first list, and the profile of the three merged banks.

In [9]:
pairs = mp.loc[mp['possible_same_as'].notna() & (mp['possible_same_as'].astype(str).str.strip() != ''),
               ['standard_name', 'possible_same_as']].drop_duplicates()
rename = dict(zip(pairs['standard_name'], pairs['possible_same_as']))
print('Merging:', rename)

d2 = df.copy()
d2['bank_name'] = d2['bank_name'].replace(rename)
print('Overlapping bank-months to combine:', d2.duplicated(['month', 'bank_name']).sum())

def wavg(g, col):
    return (g[col] * g['volume_mn']).sum() / g['volume_mn'].sum()

d2 = (d2.groupby(['month', 'bank_name'])
        .apply(lambda g: pd.Series({'volume_mn': g['volume_mn'].sum(),
                                    'approved_pct': wavg(g, 'approved_pct'),
                                    'bd_pct': wavg(g, 'bd_pct'),
                                    'td_pct': wavg(g, 'td_pct'),
                                    'is_ranked': g['is_ranked'].min()}), include_groups=False)
        .reset_index())
d2['est_failed'] = d2['volume_mn'] * d2['td_pct'] / 100
assert np.isclose(d2['volume_mn'].sum(), df['volume_mn'].sum())      # no volume lost

def top_share(d, n=10):
    r = d[d['is_ranked'] == 1].groupby('bank_name')['est_failed'].sum().sort_values(ascending=False)
    return r.head(n).sum() / r.sum() * 100

bm2 = build_bank_month(d2)
t_base, t_mrg = top_share(df), top_share(d2)
watch2 = sorted(matrix(bm2).loc[lambda x: x['quadrant'] == '1 Fix first', 'bank_name'])

print(f'Top 10 share of failures: {t_base:.1f}% -> {t_mrg:.1f}%')
print('Fix-first list unchanged:', watch2 == watch)
print('Tier sizes after merge:', bm2.groupby('tier')['bank_name'].nunique().to_dict())

cols = ['bank_name', 'months', 'weighted_td', 'persistence', 'type']
before = base_profile[base_profile['bank_name'].isin(rename.values())][cols]
after = profile(bm2)[lambda p: p['bank_name'].isin(rename.values())][cols]
print('\nBEFORE merge'); print(before.round(2).to_string(index=False))
print('\nAFTER merge');  print(after.round(2).to_string(index=False))

Merging: {'Andhra Pradesh Grameena Bank': 'Andhra Pragathi Grameena Bank', 'Rajasthan Gramin Bank': 'Rajasthan Marudhara Gramin Bank', 'Uttar Pradesh Gramin Bank': 'Baroda U.P. Bank'}
Overlapping bank-months to combine: 0
Top 10 share of failures: 70.3% -> 70.7%
Fix-first list unchanged: True
Tier sizes after merge: {'Large': 13, 'Mid': 11, 'Small': 25}

BEFORE merge
                      bank_name  months  weighted_td  persistence    type
  Andhra Pragathi Grameena Bank      14         5.43       100.00 Chronic
               Baroda U.P. Bank      20         4.13        95.00 Chronic
Rajasthan Marudhara Gramin Bank      21         2.62        95.24 Chronic

AFTER merge
                      bank_name  months  weighted_td  persistence    type
  Andhra Pragathi Grameena Bank      16         4.74       100.00 Chronic
               Baroda U.P. Bank      23         4.12        95.65 Chronic
Rajasthan Marudhara Gramin Bank      24         2.58        95.83 Chronic


## S3: With and without unranked entries

**What:** Compare the monthly weighted TD rate using all 1,198 rows against ranked banks only (the 6 unranked entries are credit-card, fintech and similar entries NPCI does not rank).

**Why:** Q1 used all rows, the bank analyses used ranked only. We must show that choice does not change the trend.

In [10]:
r_only = df[df['is_ranked'] == 1]
m_all = df.groupby('month').apply(period_rate, include_groups=False)
m_rk = r_only.groupby('month').apply(period_rate, include_groups=False)
gap = (m_all - m_rk).abs().max()

print(f'Ranked banks are {r_only["volume_mn"].sum() / df["volume_mn"].sum() * 100:.2f}% of volume')
print(f'Largest monthly gap between the two series: {gap:.4f} pp')
print(f'Year 1 / Year 2, all rows:     {y1:.3f} / {y2:.3f}')
print(f'Year 1 / Year 2, ranked only:  {period_rate(r_only[r_only["month"].isin(p1)]):.3f} / '
      f'{period_rate(r_only[r_only["month"].isin(p2)]):.3f}')

Ranked banks are 99.33% of volume
Largest monthly gap between the two series: 0.0052 pp
Year 1 / Year 2, all rows:     0.432 / 0.324
Year 1 / Year 2, ranked only:  0.434 / 0.327


## S4: A gentler what-if target

**What:** The SQL what-if let every bank-month above its tier median fall to that median. That is the most generous target. Here we allow banks to stay up to 1.25x, 1.5x or 2x above the median.

**Why:** The "62% avoidable" figure is an upper limit, not a forecast. A range is more honest and more defensible.

In [11]:
rows = []
w = bm[bm['bank_name'].isin(watch)]
for k in [1.0, 1.25, 1.5, 2.0]:
    total_k, pct_k = prize(bm, k)
    watch_k, _ = prize(w, k)
    rows.append((k, round(total_k, 1), round(pct_k, 1), round(watch_k / total_k * 100, 0)))
s4 = pd.DataFrame(rows, columns=['target_x_median', 'avoidable_mn', 'pct_of_failures', 'watchlist_share_pct'])
s4

,target_x_median,avoidable_mn,pct_of_failures,watchlist_share_pct
0,1.00,1118.3,62.1,63.0
1,1.25,1007.9,56.0,64.0
2,1.50,907.3,50.4,65.0
3,2.00,737.2,41.0,66.0


## S5: Is "Episodic" an artifact of tiny rates?

**What:** CV = standard deviation / average. When a bank's average rate is almost zero (Axis 0.03%), small wobbles give a big CV and an "Episodic" label even though its failures are tiny.

**Why:** We want the label to mean "spiky and worth attention", not "near-zero rate". We require a minimum average TD rate before a bank can be Episodic, and see who drops out.

In [12]:
s5_rows = []
for floor in [0, 0.05, 0.10, 0.20]:
    p = profile(bm, floor)
    s5_rows.append((floor, int((p['type'] == 'Episodic').sum()), int((p['type'] == 'Healthy').sum()),
                    sorted(p.loc[p['type'] == 'Episodic', 'bank_name'])))
s5 = pd.DataFrame(s5_rows, columns=['min_avg_td_pct', 'episodic', 'healthy', 'episodic_banks'])
print(s5[['min_avg_td_pct', 'episodic', 'healthy']].to_string(index=False))
print('\nStill Episodic with a 0.10 floor:', s5.loc[2, 'episodic_banks'])

 min_avg_td_pct  episodic  healthy
           0.00         9       19
           0.05         8       20
           0.10         7       21
           0.20         3       25

Still Episodic with a 0.10 floor: ['AU Small Finance Bank', 'Bank of Baroda', 'DBS Bank India Limited', 'ESAF Small Finance Bank Ltd', 'Federal Bank Ltd.', 'Kotak Mahindra Bank', 'Punjab and Sind Bank']


## Analysis 8: Spearman correlation on month-on-month changes

**What:** Do months where volume jumps also see failure-rate jumps? We correlate the **change** in volume (% vs last month) with the **change** in TD rate (pp vs last month).

**Why changes, not levels:** both volume and (to a lesser degree) failures trend, so correlating levels would show a link just because both move over time.

**Why Spearman:** it uses ranks, so extreme values (a tiny bank's huge swing) do not dominate.

**Two levels:** (1) national, 23 monthly changes; (2) bank level, pooled consecutive-month changes of the 49 eligible banks, plus each tier.

**Read it as:** descriptive only. rho near 0 = no relationship. A p-value above 0.05 means we cannot rule out chance.

In [13]:
# (1) National: 23 month-on-month changes
monthly['vol_chg_pct'] = monthly['volume'].pct_change() * 100
monthly['td_chg_pp'] = monthly['td_rate'].diff()
x = monthly.dropna()
rho, pval = spearmanr(x['vol_chg_pct'], x['td_chg_pp'])
print(f'National:   n={len(x):>4}   rho={rho:6.3f}   p={pval:.3f}')

# (2) Bank level: only pairs of CONSECUTIVE months for the same bank
idx = {m: i for i, m in enumerate(months)}
b = bm.sort_values(['bank_name', 'month']).copy()
b['mi'] = b['month'].map(idx)
g = b.groupby('bank_name')
b['gap'] = g['mi'].diff()                                   # 1 = consecutive months
b['vol_chg_pct'] = g['volume_mn'].pct_change() * 100
b['td_chg_pp'] = g['td_pct'].diff()
pool = b[b['gap'] == 1].dropna(subset=['vol_chg_pct', 'td_chg_pp'])

rho2, p2_ = spearmanr(pool['vol_chg_pct'], pool['td_chg_pp'])
print(f'All banks:  n={len(pool):>4}   rho={rho2:6.3f}   p={p2_:.3f}')
for t_, gg in pool.groupby('tier'):
    r_, p_ = spearmanr(gg['vol_chg_pct'], gg['td_chg_pp'])
    print(f'{t_:<10}  n={len(gg):>4}   rho={r_:6.3f}   p={p_:.3f}')

National:   n=  23   rho= 0.391   p=0.065
All banks:  n=1078   rho=-0.006   p=0.838
Large       n= 299   rho=-0.058   p=0.315
Mid         n= 253   rho=-0.095   p=0.134
Small       n= 526   rho= 0.055   p=0.207


## Step 9: Verdict table

**What:** One table of every finding, the check that tested it, the result and a verdict.

**Why:** This table decides what goes into the memo and resume, and it is the honest record of what we tested. Save it to the repo.

In [14]:
n_ep_base = int((base_profile['type'] == 'Episodic').sum())
n_ep_floor = int(s5.loc[2, 'episodic'])

summary = pd.DataFrame([
    ('Year on year improvement (0.432% to 0.324%)', 'S1 rounding bounds',
     f'{a_lo:.3f}-{a_hi:.3f} vs {b_lo:.3f}-{b_hi:.3f}', 'Holds' if a_lo > b_hi else 'Fragile'),
    ('Fix-first banks run 1.25x+ their tier rate', 'S1 worst case',
     f'weakest case {s1["worst_case_severity_x"].min():.2f}x',
     'Holds' if s1['worst_case_severity_x'].min() >= 1.25 else 'Fragile'),
    ('Same 6 banks in Fix first', 'S2 merge 3 pairs',
     ', '.join(sorted(set(watch2) ^ set(watch))) or 'no change', 'Holds' if watch2 == watch else 'Changes'),
    ('Top 10 hold about 70% of failures', 'S2 merge 3 pairs',
     f'{t_base:.1f}% to {t_mrg:.1f}%', 'Holds' if abs(t_mrg - t_base) < 1 else 'Changes'),
    ('Trend unaffected by unranked entries', 'S3 with/without unranked',
     f'largest monthly gap {gap:.4f} pp', 'Holds' if gap < 0.01 else 'Changes'),
    ('Avoidable failures (62% at median target)', 'S4 gentler target',
     f"{s4.loc[2, 'pct_of_failures']:.0f}% at 1.5x, {s4.loc[3, 'pct_of_failures']:.0f}% at 2x",
     'Ceiling only, quote a range'),
    ('Episodic label', 'S5 minimum rate',
     f'{n_ep_base} banks, {n_ep_floor} with a 0.10 floor', 'Partly an artifact of tiny rates'),
    ('Volume vs TD rate, national monthly', 'Spearman on MoM changes',
     f'rho {rho:.2f}, p {pval:.3f}, n {len(x)}',
     'Weak, not significant at 5%' if pval >= 0.05 else 'Significant'),
    ('Volume vs TD rate, bank level', 'Spearman on MoM changes',
     f'rho {rho2:.2f}, p {p2_:.2f}, n {len(pool)}',
     'No relationship' if p2_ >= 0.05 else 'Significant'),
], columns=['finding', 'check', 'result', 'verdict'])

print(summary.to_string(index=False))
summary.to_csv('sensitivity_summary.csv', index=False)
try:
    from google.colab import files
    files.download('sensitivity_summary.csv')                # Colab: saves to your Downloads
except ImportError:
    pass

                                    finding                    check                        result                          verdict
Year on year improvement (0.432% to 0.324%)       S1 rounding bounds    0.427-0.437 vs 0.319-0.329                            Holds
 Fix-first banks run 1.25x+ their tier rate            S1 worst case            weakest case 1.50x                            Holds
                  Same 6 banks in Fix first         S2 merge 3 pairs                     no change                            Holds
          Top 10 hold about 70% of failures         S2 merge 3 pairs                70.3% to 70.7%                            Holds
       Trend unaffected by unranked entries S3 with/without unranked largest monthly gap 0.0052 pp                            Holds
  Avoidable failures (62% at median target)        S4 gentler target        50% at 1.5x, 41% at 2x      Ceiling only, quote a range
                             Episodic label          S5 minimum rate  9 bank

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>